<br/>

<div align="center">
<span style="font-size: 2.5em;">Optuna Study Loader</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Inspect and summarize HPO results stored in SQLite</span>
</div>

## Overview

Hyperparameter optimization (HPO) is needed because many model settings (learning rate,
network size, dropout, optimizer) strongly affect performance but are **not learned**
from data. Manual tuning is slow, subjective, and hard to reproduce, especially as
models grow more complex.

Formally, we treat training as a black-box objective

$$
\min_{\lambda \in \Lambda} \; f(\lambda)
$$

where $\lambda$ are hyperparameters and $f(\lambda)$ is a validation metric (e.g., loss).
In practice, $f(\lambda)$ is estimated by training and evaluating a model:

$$
\hat{f}(\lambda) = \frac{1}{K} \sum_{k=1}^{K} \mathcal{L}\big(\theta^{(k)}(\lambda); \mathcal{D}_{\text{val}}^{(k)}\big)
$$

**Optuna** is an automated HPO framework that samples hyperparameters, trains the model,
and keeps what improves the validation metric, while pruning unpromising trials to save
compute. With pruning, a trial can be stopped early if its intermediate loss $f_t(\lambda)$
is worse than a reference statistic (e.g., the median at epoch $t$).

This notebook:
- loads an existing Optuna study
- prints study statistics and the best trial
- builds a Pandas table for filtering
- provides optional visualization diagnostics

In [1]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
desired_root_name = "xenon-sbi" 
while os.path.basename(os.getcwd()) != desired_root_name:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

import torch
import pandas as pd
import optuna
from optuna.trial import TrialState

from configs.config import PARAM_RANGES


In [2]:
# ===========================================
# PARAMETERS TO DEFINE THE DATA AND MODEL 
# ===========================================

modelname = "full"                
datatag   = "low"
label    = "signal_bg"
halo      = "default" 
n_train   = 300_000
top_k     = 10    

logm_range = PARAM_RANGES[datatag]["logm_range"]
logcp_range = PARAM_RANGES[datatag]["logcp_range"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Load Study

Create or load the study and print summary statistics.

In [3]:
# ==========================================
# LOAD MULTIPLE OPTUNA STUDIES + STATISTICS
# ==========================================

study_names = [
    "full_low",
    "ntothighest_low",
    "hist_s1s2__signal_only",
    "hist_s1s2__signal_bkg",
]

study_summaries = []
loaded_studies = {}

for study_name in study_names:
    storage_path = f"sqlite:///hpo/optuna_studies/{study_name}.db"

    study = optuna.create_study(
        study_name=study_name,
        storage=storage_path,
        load_if_exists=True,
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(),
        sampler=optuna.samplers.TPESampler(),
    )

    loaded_studies[study_name] = study

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])
    failed_trials = study.get_trials(deepcopy=False, states=[TrialState.FAIL])
    best_trial = study.best_trial

    print(f"\n{'=' * 72}")
    print(f"Study: {study_name}")
    print(f"  Number of finished trials: {len(study.trials)}")
    print(f"  Number of pruned trials:   {len(pruned_trials)}")
    print(f"  Number of complete trials: {len(complete_trials)}")
    print(f"  Number of failed trials:   {len(failed_trials)}")
    print(f"  Best value:                {best_trial.value:.6f}")
    print("  Best params:")
    for key, value in best_trial.params.items():
        print(f"    {key}: {value}")

    study_summaries.append({
        "study": study_name,
        "finished": len(study.trials),
        "pruned": len(pruned_trials),
        "complete": len(complete_trials),
        "failed": len(failed_trials),
        "best_value": best_trial.value,
    })

comparison_df = pd.DataFrame(study_summaries).sort_values("best_value", ascending=True).reset_index(drop=True)
comparison_df

[I 2026-08-25 12:55:16,545] Using an existing study with name 'full_low' instead of creating a new one.
[I 2026-08-25 12:55:16,733] Using an existing study with name 'ntothighest_low' instead of creating a new one.



Study: full_low
  Number of finished trials: 190
  Number of pruned trials:   122
  Number of complete trials: 41
  Number of failed trials:   19
  Best value:                0.378735
  Best params:
    hidden_dim: 64
    num_layers: 7
    dropout: 0.0
    lr: 0.0005
    weight_decay: 0.001
    batch_size: 4096
    lr_decay: 0.95


[I 2026-08-25 12:55:16,968] Using an existing study with name 'hist_s1s2__signal_only' instead of creating a new one.



Study: ntothighest_low
  Number of finished trials: 150
  Number of pruned trials:   46
  Number of complete trials: 49
  Number of failed trials:   29
  Best value:                0.376370
  Best params:
    hidden_dim: 512
    num_layers: 8
    dropout: 0.05
    lr: 0.003
    weight_decay: 1e-06
    batch_size: 2048
    lr_decay: 0.95

Study: hist_s1s2__signal_only


[I 2026-08-25 12:55:17,204] Using an existing study with name 'hist_s1s2__signal_bkg' instead of creating a new one.


  Number of finished trials: 176
  Number of pruned trials:   94
  Number of complete trials: 41
  Number of failed trials:   11
  Best value:                0.411448
  Best params:
    bins: 10
    batch_size: 1024
    lr_decay: 0.95
    hidden_dim: 128
    num_layers: 5
    dropout: 0.1
    lr: 0.003
    weight_decay: 0.001

Study: hist_s1s2__signal_bkg
  Number of finished trials: 232
  Number of pruned trials:   136
  Number of complete trials: 89
  Number of failed trials:   5
  Best value:                0.584949
  Best params:
    bins: 10
    batch_size: 1024
    lr_decay: 0.95
    hidden_dim: 64
    num_layers: 6
    dropout: 0.2
    lr: 0.003
    weight_decay: 0.0


,study,finished,pruned,complete,failed,best_value
0,ntothighest_low,150,46,49,29,0.376370
1,full_low,190,122,41,19,0.378735
2,hist_s1s2__signal_only,176,94,41,11,0.411448
3,hist_s1s2__signal_bkg,232,136,89,5,0.584949


## 3. Pandas Summary

Convert trials to a table for quick filtering and inspection.

In [4]:
# ==========================================
# LARGE PARAMETER COMPARISON TABLE
# ==========================================

rows = []
for study_name, study in loaded_studies.items():
    best_trial = study.best_trial
    row = {
        "study": study_name,
        "best_value": best_trial.value,
        "n_finished": len(study.trials),
    }
    row.update(best_trial.params)
    rows.append(row)

param_comparison_df = pd.DataFrame(rows)

# Order columns: metadata first, then sorted parameter columns
meta_cols = ["study", "best_value", "n_finished"]
param_cols = sorted([c for c in param_comparison_df.columns if c not in meta_cols])
param_comparison_df = param_comparison_df[meta_cols + param_cols]
param_comparison_df = param_comparison_df.sort_values("best_value", ascending=True).reset_index(drop=True)

display(param_comparison_df.style.format({"best_value": "{:.6f}"}))

,study,best_value,n_finished,batch_size,bins,dropout,hidden_dim,lr,lr_decay,num_layers,weight_decay
0,ntothighest_low,0.376370,150,2048,nan,0.050000,512,0.003000,0.950000,8,0.000001
1,full_low,0.378735,190,4096,nan,0.000000,64,0.000500,0.950000,7,0.001000
2,hist_s1s2__signal_only,0.411448,176,1024,10.000000,0.100000,128,0.003000,0.950000,5,0.001000
3,hist_s1s2__signal_bkg,0.584949,232,1024,10.000000,0.200000,64,0.003000,0.950000,6,0.000000


## 4. Visualization

Optional plots for diagnostics and interpretation.

In [5]:
if 0:
    optuna.visualization.plot_optimization_history(study)
    optuna.visualization.plot_param_importances(study)
    optuna.visualization.plot_parallel_coordinate(study)
    optuna.visualization.plot_contour(study) # (study, params=["lr", "weight_decay"])
    optuna.visualization.plot_slice(study) # (study, params=["lr", "weight_decay"])
    optuna.visualization.plot_edf(study)
    optuna.visualization.plot_intermediate_values(study)